# ChlorA Neural Network Prediction
Predicts Chlorophyll-A concentration from CalCOFI oceanographic features using a fully-connected neural network (PyTorch).

In [ ]:
%pip install torch torchvision torchaudio 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch as tf
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print('PyTorch version:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

torch.manual_seed(42)
np.random.seed(42)

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('Finalized_CalCOFI_Phytoplankton.csv', parse_dates=['Date'])
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['ChlorA'].hist(bins=60, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('ChlorA Distribution')
axes[0].set_xlabel('ChlorA')

np.log1p(df['ChlorA']).hist(bins=60, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('log1p(ChlorA) Distribution')
axes[1].set_xlabel('log1p(ChlorA)')
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing

In [ ]:
# Extract temporal features
df['Month'] = df['Date'].dt.month
df['Year']  = df['Date'].dt.year
# Encode month cyclically so Jan and Dec are close
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

FEATURE_COLS = [
    'Lat_Dec', 'Lon_Dec',
    'T_degC', 'Salnty', 'O2ml_L', 'O2Sat',
    'Phaeop', 'PO4uM', 'SiO3uM', 'NO3uM',
    'Cloud_Amt', 'NP_Ratio', 'Shore_km',
    'Month_sin', 'Month_cos', 'Year'
]
TARGET = 'ChlorA'

data = df[FEATURE_COLS + [TARGET]].dropna()
print(f'Rows after dropping NaN: {len(data):,}')

X = data[FEATURE_COLS].values
# Predict log1p(ChlorA) — makes the target more Gaussian
y = np.log1p(data[TARGET].values)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42
)
print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

## 3. Build Neural Network

In [ ]:
class ChlorANet(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout / 2),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model = ChlorANet(X_train.shape[1]).to(DEVICE)
print(model)
print(f'\nTotal params: {sum(p.numel() for p in model.parameters()):,}')

## 4. Train

In [ ]:
def to_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train, y_train)
val_loader   = to_loader(X_val,   y_val,   shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=7)
criterion = nn.HuberLoss()

EPOCHS       = 200
PATIENCE     = 15
best_val     = float('inf')
patience_ctr = 0
best_state   = None
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(Xb)
    train_losses.append(epoch_loss / len(train_loader.dataset))

    # --- validate ---
    model.eval()
    with torch.no_grad():
        vl = sum(criterion(model(Xb.to(DEVICE)), yb.to(DEVICE)).item() * len(Xb)
                 for Xb, yb in val_loader)
    val_loss = vl / len(val_loader.dataset)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | train {train_losses[-1]:.4f} | val {val_loss:.4f} | lr {optimizer.param_groups[0]["lr"]:.2e}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val:.4f}')

## 5. Evaluate

In [ ]:
model.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    y_pred_log = model(X_test_t).cpu().numpy()

y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print(f'Test RMSE : {rmse:.4f}')
print(f'Test MAE  : {mae:.4f}')
print(f'Test R²   : {r2:.4f}')

In [ ]:
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# --- Loss curves ---
ax0 = fig.add_subplot(gs[0])
ax0.plot(train_losses, label='Train loss')
ax0.plot(val_losses,   label='Val loss')
ax0.set_title('Training Loss (Huber)')
ax0.set_xlabel('Epoch')
ax0.legend()

# --- Predicted vs Actual ---
ax1 = fig.add_subplot(gs[1])
lim = max(y_true.max(), y_pred.max()) * 1.05
ax1.scatter(y_true, y_pred, alpha=0.3, s=8, color='steelblue')
ax1.plot([0, lim], [0, lim], 'r--', lw=1.5)
ax1.set_xlim(0, lim); ax1.set_ylim(0, lim)
ax1.set_title(f'Predicted vs Actual\nR²={r2:.3f}')
ax1.set_xlabel('Actual ChlorA'); ax1.set_ylabel('Predicted ChlorA')

# --- Residuals ---
ax2 = fig.add_subplot(gs[2])
residuals = y_true - y_pred
ax2.scatter(y_pred, residuals, alpha=0.3, s=8, color='darkorange')
ax2.axhline(0, color='red', lw=1.5, linestyle='--')
ax2.set_title('Residuals')
ax2.set_xlabel('Predicted ChlorA'); ax2.set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 6. Feature Importance (Permutation)

In [ ]:
def predict_log(X_np):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X_np, dtype=torch.float32).to(DEVICE)).cpu().numpy()

baseline_mse = mean_squared_error(y_test, predict_log(X_test))

importances = []
for i, col in enumerate(FEATURE_COLS):
    X_perm = X_test.copy()
    np.random.shuffle(X_perm[:, i])
    perm_mse = mean_squared_error(y_test, predict_log(X_perm))
    importances.append(perm_mse - baseline_mse)

imp_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='teal')
plt.title('Permutation Feature Importance\n(increase in log-space MSE when shuffled)')
plt.xlabel('ΔMSE')
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
import joblib

torch.save(model.state_dict(), 'chlora_nn_model.pt')
joblib.dump(scaler, 'chlora_scaler.pkl')
print('Model and scaler saved.')